In [ ]:
# ------------------- Use MATLAB engine to pass variables to python plotting easily

In [1]:

%pip install --upgrade pip
%pip install numpy matplotlib
# (optional) MATLAB engine wheel for R2025a:
%pip install matlabengine==25.1.2

C:\Users\13523\venvs\lab\Scripts\python.exe
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# ------------------- Import Libraries

In [12]:
import sys, pkgutil, matlab, matlab.engine, time, io
print("Py exe:", sys.executable)
print("matlab pkg:", matlab.__file__)
print("engine pkg:", matlab.engine.__file__)

import numpy as np
import matplotlib.pyplot as plt



Py exe: C:\Users\13523\venvs\lab\Scripts\python.exe
matlab pkg: C:\Users\13523\venvs\lab\Lib\site-packages\matlab\__init__.py
engine pkg: C:\Users\13523\venvs\lab\Lib\site-packages\matlab\engine\__init__.py


In [ ]:
# ------------------- Start MATLAB Engine

In [15]:
# Fast headless start. If this hangs, try "-desktop".
eng = matlab.engine.start_matlab("-nojvm")
print("MATLAB:", eng.eval("version", nargout=1))

MATLAB: 25.1.0.2973910 (R2025a) Update 1


In [ ]:
# ------------------- Call .m 2D Radar Scan Backprojection

In [18]:
# Point MATLAB to your code repo
eng.cd(r"C:\Repos\myrepo", nargout=0)
eng.eval("addpath(genpath(pwd))", nargout=0)

# Path to a scan in your worktree
scan_path = r"C:\Repos\YZ_Scans\scans\YZ_2D_20250811_204940.mat"

# Call your function with the scan file
eng.Backprojection2D(scan_path, nargout=0)

MatlabExecutionError: Unable to change current folder to 'C:\Repos\myrepo' (Name is nonexistent or not a folder).


In [ ]:
# ------------------- 3D Radar Scan Backprojection

In [ ]:
% y axis - horizontal printer move
% x axis - vetical printer move
% z axis - towards target

% Backprojection based off of grid center (assume NxN square grid)
%close all; clc; clear;

dataFile = '3D_20250730_193754.mat';
close all; disp('Reconstruction Starting'); tic;

%% ---------------- USER PARAMS ----------------------------------------
load(dataFile)   % contains variable `recs`
[~, baseName] = fileparts(dataFile);

vtrigU_ants_location;
[Xgrid,Ygrid,Zgrid]=meshgrid(xgrid,ygrid,zgrid);

src = reshape(cat(4,Xgrid,Ygrid,Zgrid),[],3);
src2 = permute(src,[3,2,4,1]);
c = physconst('lightspeed'); %(m/s)
N_freq = length(freq);
Nfft = 2^(ceil(log2(size(freq,2)))+1);

%% pre‑allocate reconstruction stack -------------

%length of each voxel axis
Ny = numel(xgrid);
Nx = numel(ygrid);
Nz = numel(zgrid);

nRecs   = size(recs,3);                 % how many sweeps in .mat
y_accum = zeros(Ny, Nx, Nz, nRecs, 'single');   % or 'double'
y_cart_sum = zeros(Ny, Nx, Nz,'single');
toDB = @(M) 20*log10(abs(M)+eps) - max(20*log10(abs(M(:))+eps)); %db helper function


%% Back Projection Loop --------------------------
for i=1:nRecs
X = squeeze(recs(:,:,i));
Rvec = src2-(VtrigU_ants_location + [printer_offsets(i,1) printer_offsets(i,2) 0]); %printer_offsets..  y(R->L) x(up) from vtrigU
Rmag = rssq(Rvec,2);
Rtheta = atan2(rssq(Rvec(:,1:2,:,:),2),Rvec(:,3,:,:));
Rphi = atan2(Rvec(:,2,:,:),Rvec(:,1,:,:));
Sphase = 2*pi*Rmag.*freq/c; %Electrical Length in Radians
RCS = 1; %m^2
lambda = c./freq; csf = sqrt(RCS).*lambda./((4*pi).^(3/2));
Smag = 10^(5.8/20)*RadiationPattern(Rtheta,Rphi)./Rmag;

% beam steering matrix
H2 = zeros(length(TxRxPairs),length(freq),1,length(src2));
for ii = 1:length(TxRxPairs)
    tx = TxRxPairs(ii,1); rx = TxRxPairs(ii,2);
    H2(ii,:,:,:) = 1./(csf.*Smag(tx,:,:,:).*Smag(rx,:,:,:).*...
              exp(-1j.*(Sphase(tx,:,:,:)+Sphase(rx,:,:,:))));
end
H2 = reshape(permute(H2,[4,1,2,3]),length(src2),[]); %xyz x txrx x freq

% Identify resonant frequencies 
thresh = 3;
lnconv = min(max(floor(N_freq/8)*2+1,floor(50/(freq(2)-freq(1)))*2+1),...
         floor(3*N_freq/8)*2+1); %conv length between 1/4 and 3/4 N_freq
c2 = -ones(lnconv,1)/(lnconv-1);
c2((lnconv+1)/2) = 1;

padsig = 20*log10(rssq(X,1));
%padsig = padsig(:).';  % ensure row vector

padsig = [padsig((lnconv-1)/2:-1:1),padsig,padsig(end:-1:end-(lnconv-1)/2+1)]; 
padsig = conv(padsig,c2,'valid');        
f_res = padsig>thresh;

%Remove resonant frequencies
X = X .* (1 - f_res);
%convert to complex time domain signal
x = ifft(X,Nfft,2);
       
y_cart = reshape(H2*reshape(X,[],1), numel(xgrid), numel(ygrid), numel(zgrid));
temp_y = y_cart;

% accumulate
y_accum(:,:,:,i) = temp_y;

y_cart_sum = y_cart_sum + temp_y; % accumulate for reconstruction

a = sprintf('Image %d/%d Proccessed',i, numel(recs(1,1,:)) ); disp(a);

end %% End of Backprojection

toc

In [ ]:
# ------------------- Write MATLAB variables to numpy